In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import os
import tiktoken
import time
import argparse
import itertools
from more_itertools import batched

In [2]:
%cd ..

/Users/ashutosh/personal/study/gpt


In [3]:
PADDING_VALUE=0
def collate_fn(batch):
    # list of tuples
    x, y = zip(*batch)
    x_padded = torch.nn.utils.rnn.pad_sequence(x, batch_first=True, padding_value=PADDING_VALUE)
    y_padded = torch.nn.utils.rnn.pad_sequence(y, batch_first=True, padding_value=PADDING_VALUE)
    mask = (x_padded==PADDING_VALUE).bool()
    return (x_padded, y_padded), mask

In [4]:
# I have couple of options for training data prep:
# (1) varying sequence length and (2) varying starting point subject to min and max sequence lengths

In [5]:
import itertools

class GPTDataUtils:

    @staticmethod
    def load_raw_data(path, columns=None, format='parquet'):
        if format == 'parquet':
            return pd.read_parquet(path, columns=columns)
        else:
            raise NotImplementedError()
        
class GPTDatasetSequancePacking(torch.utils.data.Dataset):
    def __init__(self, raw_data_path, num_threads=os.cpu_count(), min_seq_len=1, max_seq_len=512, min_start_idx=0):
        # load the raw data
        dataloader_start_time = time.time()
        self.raw_data_df = GPTDataUtils.load_raw_data(path=raw_data_path)#"assets/raw_data")
        dataloader_end_time = time.time()
        # print(f"time to load raw data = {round(dataloader_end_time-dataloader_start_time, 4)} seconds")

        # load the pretrained tokenizer by gpt2
        self.tokenizer = tiktoken.get_encoding(encoding_name="gpt2")

        self.raw_data_df['text'] = self.raw_data_df['text'] + '<|endoftext|>'

        batch_encode_start_time = time.time()
        self.tokens = self.tokenizer.encode_batch(text=self.raw_data_df['text'], num_threads=num_threads,  allowed_special={"<|endoftext|>"})
        unique_last_tokens = set([tokens[-1] for tokens in self.tokens])
        print(f"Unique last tokens: {unique_last_tokens}")
        assert len(unique_last_tokens) == 1, "All sequences should end with <|endoftext|> token"
        self.tokens_flattened = torch.tensor(list(itertools.chain.from_iterable(self.tokens)))

        batch_encode_end_time = time.time()
        print(f"num_thread = {num_threads} \t| raw_data_load_time = {round(dataloader_end_time-dataloader_start_time, 4)} \t| tokenizer_batch_encode_time = {round(batch_encode_end_time-batch_encode_start_time, 4)}")

        self.min_seq_len = min_seq_len
        self.max_seq_len = max_seq_len
        self.min_start_idx = min_start_idx

    def __len__(self):
        return len(self.tokens_flattened) // self.max_seq_len

    def __getitem__(self, index):
        start_idx = index*self.max_seq_len
        end_idx = (index+1)*self.max_seq_len
        if end_idx >= len(self.tokens_flattened)-1:
            start_idx = len(self.tokens_flattened) - self.max_seq_len - 1
            end_idx = len(self.tokens_flattened) - 1
        x = self.tokens_flattened[start_idx:end_idx]
        y = self.tokens_flattened[start_idx+1:end_idx+1]
        
        # mask the loss for the tokens which are after <|endoftext|> token, because those tokens are not actually seen by the model during training, and we don't want the model to learn from those tokens.
        # this will make it slow though due to finding EOT mask for each sequence
        EOT_mask = (x == self.tokenizer.eot_token)
        # print(f"EOT_Mask | x = {x[EOT_mask]} | y = {y[EOT_mask]}")
        y[EOT_mask] = -100
        return x, y

In [6]:
train_dataset = GPTDatasetSequancePacking("assets/raw_data")
train_dataloader = torch.utils.data.DataLoader(
    dataset=train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    # collate_fn=collate_fn
)

Unique last tokens: {50256}
num_thread = 12 	| raw_data_load_time = 0.8862 	| tokenizer_batch_encode_time = 23.1116


In [7]:
train_dataset[2]

(tensor([  286,  6686,    13,   383, 15760,    11,   900,   510,   739,  1811,
         29804,    11,   373,   257, 11858,   437,   284,   262,  1178,   508,
           547,  9670,   284,   423,   587,  3181,   612,    13,   198,   198,
          9781,  1202,  5407, 19090,    13,  5215,    13,  9223,   741,  8835,
           382,    11,   508,  2957,  8259,  4040,   329, 16004, 33251,   287,
          5075,    11,   531,   262, 24663,   286,   262, 15760,   338,  3315,
          3085,   373, 28371,    70, 21911,    13,   198,   198,     1, 18243,
           290,  9992,  1276, 19997,  2324,   553, 21071,  2634,   531,    13,
           366,    40,  1053,  1239,  1775,  1997,   588,   428,   878,   287,
           616,  1204,    13,  1119,   761,   284,   582,   510,   290,   651,
           736,   287,   612,   526,   198,   198, 29478,   273,  2634,  9859,
         30614,  1022,   262, 13574,   287,   968, 12255,    11, 13340,    11,
           290,   287,  4347,    12,   559,    12, 3

In [8]:
temp = next(iter(train_dataloader))

In [9]:
train_dataset.tokenizer._special_tokens

{'<|endoftext|>': 50256}

In [10]:
EOT_mask = (temp[0] == train_dataset.tokenizer._special_tokens.get("<|endoftext|>"))

In [11]:
(EOT_mask).sum()

tensor(9)

In [12]:
temp[0][EOT_mask] 

tensor([50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256])

In [13]:
temp[1][EOT_mask] 

tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100])

In [14]:
temp[1][EOT_mask] 

tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100])

In [15]:
temp1_cloned = temp[1].clone()

In [16]:
temp[1][EOT_mask] = -100

In [17]:
temp1_cloned[EOT_mask]

tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100])

In [18]:
(temp[1] != temp1_cloned).sum()

tensor(0)

In [19]:
train_dataset.tokenizer.eot_token

50256

In [21]:
torch.nn.Parameter(torch.empty(1,4))

Parameter containing:
tensor([[0., 0., 0., 0.]], requires_grad=True)

In [238]:
import collections

In [239]:
l = torch.nn.Sequential(collections.OrderedDict({
    "layer1":torch.nn.Linear(8,4),
    "layer2":torch.nn.Linear(8,4),
    
})
#     torch.nn.Linear(8,4),
#     torch.nn.Linear(4,1)
)
l

Sequential(
  (layer1): Linear(in_features=8, out_features=4, bias=True)
  (layer2): Linear(in_features=8, out_features=4, bias=True)
)

In [246]:
l.layer1.bias

Parameter containing:
tensor([ 0.0979, -0.1817,  0.3176,  0.1464], requires_grad=True)

In [244]:
print(list(l.layer1.named_parameters()))

[('weight', Parameter containing:
tensor([[-0.0128, -0.1449,  0.0823, -0.0017, -0.1491,  0.1941, -0.2384, -0.1816],
        [-0.3182,  0.2329, -0.3111, -0.1454,  0.3236,  0.2434,  0.2417,  0.2817],
        [-0.0764,  0.3382,  0.0622,  0.3354, -0.2824, -0.3347,  0.2286,  0.1896],
        [-0.2172,  0.2560, -0.1943,  0.3389,  0.3018, -0.3043, -0.0617, -0.2485]],
       requires_grad=True)), ('bias', Parameter containing:
tensor([ 0.0979, -0.1817,  0.3176,  0.1464], requires_grad=True))]


In [224]:
print(list(l.parameters()))

[Parameter containing:
tensor([[-0.1818, -0.0480, -0.0981,  0.1913,  0.1307,  0.2129,  0.1312, -0.2221],
        [ 0.1688, -0.3417, -0.0569, -0.1225,  0.2704,  0.2502,  0.3284,  0.0017],
        [ 0.3490, -0.3070, -0.1383, -0.0881, -0.1468, -0.2384,  0.1319, -0.2653],
        [ 0.0548,  0.1519, -0.3352,  0.1408,  0.0181,  0.0287, -0.2870,  0.2335]],
       requires_grad=True), Parameter containing:
tensor([ 0.1772, -0.0438, -0.2684, -0.1827], requires_grad=True), Parameter containing:
tensor([[0.0058, 0.3586, 0.1696, 0.2401]], requires_grad=True), Parameter containing:
tensor([-0.2047], requires_grad=True)]


In [236]:
list(l.named_modules())

[('',
  Sequential(
    (0): Linear(in_features=8, out_features=4, bias=True)
    (1): Linear(in_features=4, out_features=1, bias=True)
  )),
 ('0', Linear(in_features=8, out_features=4, bias=True)),
 ('1', Linear(in_features=4, out_features=1, bias=True))]

In [229]:
print(list(l.modules()))

[Sequential(
  (0): Linear(in_features=8, out_features=4, bias=True)
  (1): Linear(in_features=4, out_features=1, bias=True)
), Linear(in_features=8, out_features=4, bias=True), Linear(in_features=4, out_features=1, bias=True)]


In [ ]:
for parameter in l.parameters():
    parameter.data.mul_(2)
print(list(l.parameters()))

AttributeError: 'generator' object has no attribute 'data'

In [198]:
p.data

tensor([[ 0.0562, -0.1399, -0.2960, -0.1041,  0.1357, -0.3512, -0.3461,  0.3096],
        [-0.1498, -0.3165, -0.2251, -0.0755, -0.1265, -0.1056, -0.3176, -0.3148],
        [ 0.0171,  0.3444, -0.1097, -0.2961,  0.0015, -0.2372,  0.2544, -0.2896],
        [-0.0541, -0.0885, -0.1736,  0.1383, -0.0656, -0.2253,  0.0291, -0.1993]])

In [199]:
p.data.mul_(2)

tensor([[ 0.1125, -0.2799, -0.5920, -0.2083,  0.2714, -0.7024, -0.6922,  0.6192],
        [-0.2995, -0.6329, -0.4503, -0.1509, -0.2530, -0.2112, -0.6351, -0.6296],
        [ 0.0341,  0.6888, -0.2194, -0.5922,  0.0030, -0.4744,  0.5089, -0.5791],
        [-0.1083, -0.1770, -0.3472,  0.2767, -0.1311, -0.4506,  0.0581, -0.3986]])

In [261]:
mha = torch.nn.MultiheadAttention(
    embed_dim=16, 
    num_heads=2,
    kdim=8,
    vdim=8,
    dropout=0.2,
    bias=True
)
print(list(mha.named_parameters()))

[('q_proj_weight', Parameter containing:
tensor([[ 0.3044,  0.0738,  0.0034, -0.1082,  0.3632, -0.2333,  0.0413,  0.1378,
         -0.3281,  0.3369, -0.0336,  0.3903,  0.4210, -0.2247,  0.0271, -0.2875],
        [-0.1274,  0.2641, -0.0947,  0.0890, -0.0224, -0.1920,  0.0466, -0.3994,
         -0.2412,  0.0010,  0.2459, -0.2843, -0.2615,  0.1372,  0.2654, -0.1575],
        [ 0.1162,  0.3645, -0.2636, -0.3106, -0.4067, -0.3695, -0.0261, -0.0407,
         -0.2880, -0.2561, -0.0086, -0.1196, -0.2070,  0.1687,  0.2586,  0.2080],
        [-0.3740, -0.4277, -0.0601, -0.0914,  0.0927, -0.0289, -0.0689, -0.3558,
          0.0831, -0.2282,  0.1748, -0.3200, -0.3285, -0.0413,  0.0330,  0.2003],
        [ 0.1481, -0.2431,  0.4010,  0.0343, -0.3353, -0.0369, -0.2900,  0.3547,
          0.2981, -0.2053,  0.3300, -0.4266, -0.1252, -0.1072, -0.2922, -0.0205],
        [-0.1020,  0.0775, -0.3055,  0.3060, -0.4062, -0.1879, -0.2425, -0.2616,
          0.1666, -0.0948, -0.3844, -0.3234,  0.2654,  0.1588, 

In [262]:
mha(torch.rand(4,2,16), torch.rand(4,2,8), torch.rand(4,2,8))[1].shape

torch.Size([2, 4, 4])

In [ ]:
from gpt.modules.embedding.sinusoidal import SinusoidalPositionalEmbeddings
class CustomLayerNorm(torch.nn.Module):
    def __init__(self, d_model, eps=1e-5, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.d_model = d_model
        self.weight = torch.nn.Parameter(data=torch.ones(size=(self.d_model,)))
        self.bias = torch.nn.Parameter(data=torch.zeros(size=(self.d_model,)))
        self.eps = eps

    def forward(self, x: torch.Tensor):
        mean = x.mean(dim=-1, keepdim=True)
        # unbiased=False because we want to divide by N instead of N-1, because we are calculating the variance for the entire population (the sequence) and not a sample of the population.
        # torch by default uses unbiased=True, which divides by N-1, so we need to set it to False to divide by N. Leading to incorrect results and mismatch with torch.nn.LayerNorm results.
        # MISTAKE - initially I had not set it to false there for results weren't matching with torch.nn.LayerNorm
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_normalized = (x-mean)/torch.sqrt(var + self.eps)
        x_normalized = (x_normalized*self.weight)+self.bias
        return x_normalized


class TransformerBlock(torch.nn.Module):

    def __init__(self, d_model, n_heads, attention_dropout=0.2, scaling_factor=1, *args, **kwargs):
        """
        """
        self.d_model = d_model
        self.n_heads = n_heads
        self.MHA = torch.nn.MultiheadAttention(
            embed_dim=self.d_model, 
            num_heads=self.n_heads,
            dropout=attention_dropout,
            bias=True
        )
        self.FFN = torch.nn.Sequential(collections.OrderedDict([
            ("linear_expansion", torch.nn.Linear(in_features=d_model, out_features=4*d_model, bias=True)),
            # Mistake - I had initially forgotten the activation layer
            ("activation", torch.nn.GELU()),
            ("linear_projection", torch.nn.Linear(in_features=4*d_model, out_features=d_model, bias=True))
        ]))

        # scale MHA parameters by scaling factor
        # This was introduced in GPT2 paper to stabilize deep layers
        # ""We scale the weights of residual layers at initialization by a factor of 1/√N where N is the number of residual layers.""

        ########################################################
        # I just realised that, this is wrong AS IT is scaling all the parameters including biases which should not be scaled.
        # Because, weights gets multiplied with prev. layer's output so they contribute in scaling variance but bias just get added so it doesn't contribute
        ########################################################
        # for parameter in self.MHA.parameters():
        #     parameter.data.mul_(scaling_factor)
        # for parameter in self.FFN.parameters():
        #     parameter.data.mul_(scaling_factor)

        # scale only weights correctly
        # MISTAKE - Earlier I had scaled only in_proj_weight practically limiting the scenario where MHA is using separate q_proj_weight, k_proj_weight and v_proj_weight instead of combined in_proj_weight. This is because in some versions of PyTorch, MultiheadAttention uses separate projection weights for query, key and value instead of combined projection weight.
        if self.MHA.in_proj_weight:
            self.MHA.in_proj_weight.data.mul_(scaling_factor)
        if self.MHA.q_proj_weight:
            self.MHA.q_proj_weight.data.mul_(scaling_factor)
        if self.MHA.k_proj_weight:
            self.MHA.k_proj_weight.data.mul_(scaling_factor)
        if self.MHA.v_proj_weight:
            self.MHA.v_proj_weight.data.mul_(scaling_factor)
        self.MHA.out_proj.weight.data.mul_(scaling_factor)
        self.FFN.linear_expansion.weight.data.mul_(scaling_factor)
        self.FFN.linear_projection.weight.data.mul_(scaling_factor)
        
        self.layer_norm_mha = CustomLayerNorm(d_model=self.d_model)
        self.layer_norm_ffn = CustomLayerNorm(d_model=self.d_model)


    def forward(self, x):
        x_layer_norm_mha = self.layer_norm_mha(x)
        # MISTAKE - initially I had not provided separate query, key and values arguments. Also, I wasn't selecting first value
        # This was because in my custom implementation of MHA, I wasn't taking three inputs in MHA.
        x_post_mha = x + self.MHA(query=x_layer_norm_mha, key=x_layer_norm_mha, value=x_layer_norm_mha)[0]

        x_layer_norm_ffn = self.layer_norm_ffn(x_post_mha)
        x_post_ffn = x_post_mha + self.FFN(x_layer_norm_ffn)
        return x_post_ffn
        

class GPT2Model(torch.nn.Module):
    def __init__(self, d_model, n_heads, n_layers, vocab_size):
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.vocab_size = vocab_size
        self.transformer_layers = torch.nn.Sequential()
        self.final_layer_norm = CustomLayerNorm(d_model=self.d_model)
        self.embedding = torch.nn.Embedding(
            num_embeddings=self.vocab_size,
            embedding_dim=self.d_model,
            padding_idx=None,
            max_norm=1,
            norm_type=2
        )
        self.position_embedding = SinusoidalPositionalEmbeddings(d_model=self.d_model)
        # add sequential layers
        for layer in self.n_layers:
            self.transformer_layers.append(TransformerBlock(d_model=self.d_model, n_heads=self.n_heads, scaling_factor=1/np.sqrt(self.n_layers)))
        # add final layer normalization
        self.transformer_layers.append(self.final_layer_norm)
        # predict token with softmax
        self.transformer_layers.append(torch.nn.Linear(in_features=self.d_model, out_features=self.vocab_size))


    def forward(self, x, return_proba = False):
        x_learnt_embeddings = self.embedding(x)
        x_pos_embeddings = self.position_embedding(x)
        x_embeddings = x_learnt_embeddings + x_pos_embeddings
        x_logits = self.transformer_layers(x_embeddings)
        if return_proba:
            return torch.nn.functional.softmax(input=x_logits, dim=-1)
        else:
            return x_logits

In [ ]:
torch.nn.Sequential

In [184]:
# s = torch.nn.Sequential()
s.append(torch.nn.Linear(5,2))

Sequential(
  (0): Linear(in_features=5, out_features=2, bias=True)
  (1): Linear(in_features=5, out_features=2, bias=True)
)

In [178]:
x = torch.rand(size=(4,8,16))
x.shape

torch.Size([4, 8, 16])

In [179]:
torch.isclose(
    input=torch.nn.LayerNorm(16)(x),
    other=CustomLayerNorm(d_model=16)(x),
    rtol=1e-6,
    atol=1e-6
).all()

tensor(True)

In [149]:
class CustomLayerNorm(torch.nn.Module):
    def __init__(self, d_model, eps=1e-5, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.d_model = d_model
        self.weight = torch.nn.Parameter(data=torch.ones(size=self.d_model))
        self.bias = torch.nn.Parameter(data=torch.zeros(size=self.d_model))
        self.eps = eps

    def forward(self, x: torch.Tensor):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_normalized = (x-mean)/torch.sqrt(var + self.eps)
        x_normalized = (x_normalized*self.weight)+self.bias
        return x_normalized

In [23]:
import torch
import torch.nn as nn

# Input tensor
a = torch.tensor(
    [[[2.0, 7.0], [6.0, 4.0], [6.0, 5.0]],  # Sample 1
     [[1.0, 5.0], [7.0, 2.0], [3.0, 4.0]],  # Sample 2
     [[2.0, 3.0], [6.0, 1.0], [4.0, 5.0]],  # Sample 3
     [[3.0, 6.0], [5.0, 7.0], [2.0, 1.0]]],  # Sample 4
    dtype=torch.float32
)

# Define LayerNorm
seq_len, emb_dim = a.shape[1], a.shape[2]
layer_norm = nn.LayerNorm(emb_dim)  # LayerNorm applies normalization over last dimension (embedding dim)

# Perform LayerNorm
layer_normalized = layer_norm(a)

# Retrieve parameters
weight = layer_norm.weight  # Learnable scaling parameter (γ)
bias = layer_norm.bias      # Learnable bias parameter (β)



# Print the retrieved values
print("batch shape:\n", a.shape)
# Calculate mean and variance manually (along the last dimension)
mean = a.mean(dim=(1, 2)) 
variance = a.var(dim=(1, 2), unbiased=False)  # Compute variance along embedding dimension
print("\nMean (computed along the embedding dimension):\n", mean.shape)
print("\nVariance (computed along the embedding dimension):\n", variance.shape)


print("\nWeight (γ):\n", weight)
print("\nBias (β):\n", bias)


batch shape:
 torch.Size([4, 3, 2])

Mean (computed along the embedding dimension):
 torch.Size([4])

Variance (computed along the embedding dimension):
 torch.Size([4])

Weight (γ):
 Parameter containing:
tensor([1., 1.], requires_grad=True)

Bias (β):
 Parameter containing:
tensor([0., 0.], requires_grad=True)


In [26]:
layer_norm.weight

Parameter containing:
tensor([1., 1.], requires_grad=True)

In [28]:
list(layer_norm.parameters())

[Parameter containing:
 tensor([1., 1.], requires_grad=True),
 Parameter containing:
 tensor([0., 0.], requires_grad=True)]

In [30]:
layer_norm.weight,layer_norm.bias

(Parameter containing:
 tensor([1., 1.], requires_grad=True),
 Parameter containing:
 tensor([0., 0.], requires_grad=True))

In [37]:
a.mean(dim=(0, 1)), a.var(dim=(0, 1)), 

(tensor([3.9167, 4.1667]), tensor([4.0833, 4.3333]))

In [52]:
a.shape

torch.Size([4, 3, 2])

In [56]:
a.mean(dim=-1).unsqueeze(-1).shape

torch.Size([4, 3, 1])

In [94]:
a.mean(dim=(0,1)).unsqueeze(0).unsqueeze(0).shape

torch.Size([1, 1, 2])

In [95]:
((a-a.mean(dim=(0,1)).unsqueeze(0).unsqueeze(0))/torch.sqrt(a.var(dim=(0,1))+layer_norm.eps).unsqueeze(0).unsqueeze(0))

tensor([[[-0.9485,  1.3611],
         [ 1.0310, -0.0801],
         [ 1.0310,  0.4003]],

        [[-1.4434,  0.4003],
         [ 1.5259, -1.0408],
         [-0.4536, -0.0801]],

        [[-0.9485, -0.5604],
         [ 1.0310, -1.5212],
         [ 0.0412,  0.4003]],

        [[-0.4536,  0.8807],
         [ 0.5361,  1.3611],
         [-0.9485, -1.5212]]])

In [ ]:
mean=tensor([[[4.5000],
         [5.0000],
         [5.5000]],

        [[3.0000],
         [4.5000],
         [3.5000]],

        [[2.5000],
         [3.5000],
         [4.5000]],

        [[4.5000],
         [6.0000],
         [1.5000]]]) | 

var=tensor([[[6.2500],
         [1.0000],
         [0.2500]],

        [[4.0000],
         [6.2500],
         [0.2500]],

        [[0.2500],
         [6.2500],
         [0.2500]],

        [[2.2500],
         [1.0000],
         [0.2500]]])

In [114]:
a.var(dim=2, unbiased=False).unsqueeze(-1)

tensor([[[6.2500],
         [1.0000],
         [0.2500]],

        [[4.0000],
         [6.2500],
         [0.2500]],

        [[0.2500],
         [6.2500],
         [0.2500]],

        [[2.2500],
         [1.0000],
         [0.2500]]])

In [104]:
a.mean(dim=2)

tensor([[4.5000, 5.0000, 5.5000],
        [3.0000, 4.5000, 3.5000],
        [2.5000, 3.5000, 4.5000],
        [4.5000, 6.0000, 1.5000]])

In [133]:
((a-a.mean(dim=2).unsqueeze(-1))/torch.sqrt(a.var(dim=2, unbiased=False)+layer_norm.eps).unsqueeze(-1))

tensor([[[-1.0000,  1.0000],
         [ 1.0000, -1.0000],
         [ 1.0000, -1.0000]],

        [[-1.0000,  1.0000],
         [ 1.0000, -1.0000],
         [-1.0000,  1.0000]],

        [[-1.0000,  1.0000],
         [ 1.0000, -1.0000],
         [-1.0000,  1.0000]],

        [[-1.0000,  1.0000],
         [-1.0000,  1.0000],
         [ 1.0000, -1.0000]]])

In [ ]:
((a-a.mean(dim=-1).unsqueeze(-1))/torch.sqrt(a.var(dim=-1)+layer_norm.eps).unsqueeze(-1))

tensor([[[-0.7071,  0.7071],
         [ 0.7071, -0.7071],
         [ 0.7071, -0.7071]],

        [[-0.7071,  0.7071],
         [ 0.7071, -0.7071],
         [-0.7071,  0.7071]],

        [[-0.7071,  0.7071],
         [ 0.7071, -0.7071],
         [-0.7071,  0.7071]],

        [[-0.7071,  0.7071],
         [-0.7071,  0.7071],
         [ 0.7071, -0.7071]]])

In [ ]:
(a-a.mean(dim=(0, 1)))/torch.sqrt(a.var(dim=(0, 1))+layer_norm.eps)

RuntimeError: The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1

In [105]:
def custom_layer_norm(x: torch.Tensor, normalized_shape: torch.Size, eps: float = 1e-5) -> torch.Tensor:
    # Determine the dimensions over which to calculate mean and variance
    # PyTorch's LayerNorm normalizes over the last D dimensions
    dim = tuple(range(-len(normalized_shape), 0))
    print(f"dim = {dim}")
    
    # Calculate mean and variance over the specified dimensions
    mean = torch.mean(x, dim=dim, keepdim=True)
    # PyTorch's torch.var defaults to an unbiased estimator, 
    # but LayerNorm uses the biased estimator (correction=0 or unbiased=False)
    var = torch.var(x, dim=dim, keepdim=True, unbiased=False) 

    print(f"mean.shape={mean.shape} | var.shape={var.shape}")
    print(f"mean={mean} | var={var}")
        
    
    # Normalize the input
    x_normalized = (x - mean) / torch.sqrt(var + eps)
    
    # NOTE: This simple function does not include the learnable gamma and beta parameters
    # which are standard in the torch.nn.LayerNorm module unless disabled.
    return x_normalized

custom_layer_norm(x=a, normalized_shape=torch.Size([a.shape[2]]), eps=1e-5)

dim = (-1,)
mean.shape=torch.Size([4, 3, 1]) | var.shape=torch.Size([4, 3, 1])
mean=tensor([[[4.5000],
         [5.0000],
         [5.5000]],

        [[3.0000],
         [4.5000],
         [3.5000]],

        [[2.5000],
         [3.5000],
         [4.5000]],

        [[4.5000],
         [6.0000],
         [1.5000]]]) | var=tensor([[[6.2500],
         [1.0000],
         [0.2500]],

        [[4.0000],
         [6.2500],
         [0.2500]],

        [[0.2500],
         [6.2500],
         [0.2500]],

        [[2.2500],
         [1.0000],
         [0.2500]]])


tensor([[[-1.0000,  1.0000],
         [ 1.0000, -1.0000],
         [ 1.0000, -1.0000]],

        [[-1.0000,  1.0000],
         [ 1.0000, -1.0000],
         [-1.0000,  1.0000]],

        [[-1.0000,  1.0000],
         [ 1.0000, -1.0000],
         [-1.0000,  1.0000]],

        [[-1.0000,  1.0000],
         [-1.0000,  1.0000],
         [ 1.0000, -1.0000]]])

In [87]:
custom_layer_norm(x=a, normalized_shape=torch.Size([a.shape[2]]), eps=1e-5).std()

tensor(1.0215)

In [44]:
# torch.Size([a.shape[2]])
tuple(range(-len(torch.Size([a.shape[2]])), 0))

(-1,)

In [35]:
layer_normalized

tensor([[[-1.0000,  1.0000],
         [ 1.0000, -1.0000],
         [ 1.0000, -1.0000]],

        [[-1.0000,  1.0000],
         [ 1.0000, -1.0000],
         [-1.0000,  1.0000]],

        [[-1.0000,  1.0000],
         [ 1.0000, -1.0000],
         [-1.0000,  1.0000]],

        [[-1.0000,  1.0000],
         [-1.0000,  1.0000],
         [ 1.0000, -1.0000]]], grad_fn=<NativeLayerNormBackward0>)

In [131]:
# Input tensor
a = torch.tensor(
    [[[2.0, 7.0], [6.0, 4.0], [6.0, 5.0]],  # Sample 1
     [[1.0, 5.0], [7.0, 2.0], [3.0, 4.0]],  # Sample 2
     [[2.0, 3.0], [6.0, 1.0], [4.0, 5.0]],  # Sample 3
     [[3.0, 6.0], [5.0, 7.0], [2.0, 1.0]]],  # Sample 4
    dtype=torch.float32
)

# Define LayerNorm
seq_len, emb_dim = a.shape[1], a.shape[2]
layer_norm = nn.LayerNorm(emb_dim)  # LayerNorm applies normalization over last dimension (embedding dim)

# Perform LayerNorm
layer_normalized = layer_norm(a)
layer_normalized

tensor([[[-1.0000,  1.0000],
         [ 1.0000, -1.0000],
         [ 1.0000, -1.0000]],

        [[-1.0000,  1.0000],
         [ 1.0000, -1.0000],
         [-1.0000,  1.0000]],

        [[-1.0000,  1.0000],
         [ 1.0000, -1.0000],
         [-1.0000,  1.0000]],

        [[-1.0000,  1.0000],
         [-1.0000,  1.0000],
         [ 1.0000, -1.0000]]], grad_fn=<NativeLayerNormBackward0>)

In [96]:
import torch
import torch.nn as nn

# Define batch size, sequence length, and hidden dimension
batch_size, seq_len, hidden_dim = 2, 5, 8

# Create a sample input tensor
x = torch.randn(batch_size, seq_len, hidden_dim)

# Initialize the LayerNorm module, specifying the dimension(s) to normalize over.
# In this case, we normalize over the last dimension (hidden_dim) for each sample.
layer_norm = nn.LayerNorm(hidden_dim)

# Apply LayerNorm
output_ln = layer_norm(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output_ln.shape}")
print(f"Output mean (approx): {output_ln.mean()}")
print(f"Output variance (approx): {output_ln.var(unbiased=False)}")


Input shape: torch.Size([2, 5, 8])
Output shape: torch.Size([2, 5, 8])
Output mean (approx): 1.7881394143159923e-08
Output variance (approx): 0.9999852180480957


In [97]:
x

tensor([[[-2.7919,  0.4647,  0.7015, -0.7799,  0.9902, -2.6069, -1.7693,
           1.3293],
         [ 0.7410, -1.2647, -0.6881,  0.1010, -2.2989, -0.2290, -0.4514,
          -0.3122],
         [-0.4682,  0.3000,  1.1085,  0.6705, -0.9228,  0.7656, -0.2219,
          -1.9142],
         [ 0.4262,  0.5681, -0.9779, -0.2951,  0.6799, -0.7392, -0.6762,
          -0.3572],
         [-0.1577, -0.3712, -0.4022, -0.3498,  0.8684,  0.9525,  0.4036,
           1.4468]],

        [[-0.4462, -1.0183,  0.8743,  1.3138,  0.9109,  0.7761,  0.3036,
           0.3141],
         [ 2.1816,  0.3323, -1.7513,  0.8511, -0.5803, -0.0961,  0.7387,
           0.5691],
         [-0.6100,  2.5253,  1.7414,  0.6746,  1.0358,  0.8217,  0.3754,
          -0.2088],
         [ 0.2569,  0.2008, -1.4244,  1.2580, -1.4979,  0.5204, -0.9521,
          -0.7802],
         [ 1.3533, -0.0534,  1.3695, -0.3329, -0.4404, -0.2240,  0.2228,
          -0.5327]]])

In [100]:
output_ln.std()

tensor(1.0063, grad_fn=<StdBackward0>)

In [127]:
import torch
import torch.nn as nn

# Parameters: (batch_size, sequence_length, embedding_dim)
batch, seq, embed = 2, 5, 8
x = torch.randn(batch, seq, embed)

# Initialize LayerNorm for the last dimension
layer_norm = nn.LayerNorm(embed)

# Forward pass
output = layer_norm(x)


In [119]:
layer_norm.weight,layer_norm.bias,

(Parameter containing:
 tensor([1., 1., 1., 1., 1., 1., 1., 1.], requires_grad=True),
 Parameter containing:
 tensor([0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True))

In [138]:
x.shape

torch.Size([2, 5, 8])

In [137]:
x.mean(dim=-1)

tensor([[ 0.2993,  0.1192, -0.0978,  0.0609,  0.0447],
        [ 0.0828,  0.1763, -0.0304,  0.3003,  0.0777]])

In [136]:
(x-x.mean(dim=-1).unsqueeze(-1))/torch.sqrt(x.var(dim=-1, unbiased=False).unsqueeze(-1)+layer_norm.eps)

tensor([[[ 1.5625, -0.2364, -0.9733, -0.2095,  1.6223,  0.0235, -0.5211,
          -1.2679],
         [-0.6087, -0.7016,  1.6331, -1.8964,  0.6051,  0.2891,  0.6506,
           0.0290],
         [ 0.3729, -0.9595,  0.7846, -1.3218, -0.9294, -0.3839,  0.6723,
           1.7648],
         [ 0.4268,  0.5420,  0.3831,  1.2519, -1.8535, -1.4445,  0.5007,
           0.1935],
         [-0.3392,  1.0845, -1.7650, -0.7252,  0.3411, -0.5522,  0.3724,
           1.5836]],

        [[ 0.7667,  0.5253, -1.1107,  0.7525, -0.0114, -0.5008,  1.3698,
          -1.7914],
         [-1.1945, -0.7049,  0.7511,  0.0849,  0.3054,  1.3107,  1.0542,
          -1.6069],
         [-0.8782,  0.5536, -0.9446, -0.1439, -0.8247, -0.8121,  1.6216,
           1.4283],
         [-0.1728,  1.3168, -0.2329, -0.6494, -1.6025,  1.5646, -0.7114,
           0.4876],
         [ 0.8390, -0.6941, -1.5797, -0.1681, -0.5725,  1.7848, -0.3967,
           0.7873]]])

In [128]:
output

tensor([[[ 1.5625, -0.2364, -0.9733, -0.2095,  1.6223,  0.0235, -0.5211,
          -1.2679],
         [-0.6087, -0.7016,  1.6331, -1.8964,  0.6051,  0.2891,  0.6506,
           0.0290],
         [ 0.3729, -0.9595,  0.7846, -1.3218, -0.9294, -0.3839,  0.6723,
           1.7648],
         [ 0.4268,  0.5420,  0.3831,  1.2519, -1.8535, -1.4445,  0.5007,
           0.1935],
         [-0.3392,  1.0845, -1.7650, -0.7252,  0.3411, -0.5522,  0.3724,
           1.5836]],

        [[ 0.7667,  0.5253, -1.1107,  0.7525, -0.0114, -0.5008,  1.3698,
          -1.7914],
         [-1.1945, -0.7049,  0.7511,  0.0849,  0.3054,  1.3107,  1.0542,
          -1.6069],
         [-0.8782,  0.5536, -0.9446, -0.1439, -0.8247, -0.8121,  1.6216,
           1.4283],
         [-0.1728,  1.3168, -0.2329, -0.6494, -1.6025,  1.5646, -0.7114,
           0.4876],
         [ 0.8390, -0.6941, -1.5797, -0.1681, -0.5725,  1.7848, -0.3967,
           0.7873]]], grad_fn=<NativeLayerNormBackward0>)

In [141]:
# # Initialize LayerNorm for the last dimension
# layer_norm = nn.LayerNorm(embed)(x)

# # Forward pass
# output = layer_norm(x)
nn.LayerNorm(embed)(x)

tensor([[[ 1.5625, -0.2364, -0.9733, -0.2095,  1.6223,  0.0235, -0.5211,
          -1.2679],
         [-0.6087, -0.7016,  1.6331, -1.8964,  0.6051,  0.2891,  0.6506,
           0.0290],
         [ 0.3729, -0.9595,  0.7846, -1.3218, -0.9294, -0.3839,  0.6723,
           1.7648],
         [ 0.4268,  0.5420,  0.3831,  1.2519, -1.8535, -1.4445,  0.5007,
           0.1935],
         [-0.3392,  1.0845, -1.7650, -0.7252,  0.3411, -0.5522,  0.3724,
           1.5836]],

        [[ 0.7667,  0.5253, -1.1107,  0.7525, -0.0114, -0.5008,  1.3698,
          -1.7914],
         [-1.1945, -0.7049,  0.7511,  0.0849,  0.3054,  1.3107,  1.0542,
          -1.6069],
         [-0.8782,  0.5536, -0.9446, -0.1439, -0.8247, -0.8121,  1.6216,
           1.4283],
         [-0.1728,  1.3168, -0.2329, -0.6494, -1.6025,  1.5646, -0.7114,
           0.4876],
         [ 0.8390, -0.6941, -1.5797, -0.1681, -0.5725,  1.7848, -0.3967,
           0.7873]]], grad_fn=<NativeLayerNormBackward0>)

In [145]:
import torch
import torch.nn as nn

# Define BatchNorm for 10 features
bn = nn.BatchNorm1d(num_features=10)

# Gamma (scale) is stored in .weight
# Beta (shift) is stored in .bias
print(f"Gamma shape: {bn.weight.shape}")  # torch.Size([10])
print(f"Beta shape:  {bn.bias.shape}")    # torch.Size([10])

# Create a random batch (Batch Size = 4, Features = 10)
x = torch.randn(4, 10)
output = bn(x)

print(f"\nInput shape:  {x.shape}")       # torch.Size([4, 10])
print(f"Output shape: {output.shape}")    # torch.Size([4, 10])


Gamma shape: torch.Size([10])
Beta shape:  torch.Size([10])

Input shape:  torch.Size([4, 10])
Output shape: torch.Size([4, 10])


In [ ]:
import torch
import torch.nn as nn

# Input: Batch=4, Channels=16, Height=8, Width=8
x = torch.randn(4, 16, 8, 8)

# Initialize BatchNorm2d for 16 channels
bn = nn.BatchNorm2d(num_features=16)

# Inspect shapes
print(f"Input shape: {x.shape}")
print(f"Gamma (weight) shape: {bn.weight.shape}") # torch.Size([16])
print(f"Beta (bias) shape:    {bn.bias.shape}")   # torch.Size([16])

# Run the forward pass
output = bn(x)
print(f"Output shape: {output.shape}")            # torch.Size([4, 16, 8, 8])
